Este projeto tem como objetivo prever a avaliação que um utilizador atribuiria a um determinado filme, com base no histórico de avaliações do dataset MovieLens. Para isso, serão comparados os modelos de regressão Ridge e Árvore de Decisão.

### 1 - Carregamento e Exploração de Dados

In [26]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

display(ratings.head())
display(movies.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [27]:
print("Numero de avaliacoes:",len(ratings))
print("Numero de utilizadores:",ratings["userId"].nunique())
print("Numero de filmes no catalogo:",len(movies))

Numero de avaliacoes: 100836
Numero de utilizadores: 610
Numero de filmes no catalogo: 9742


In [28]:
print("Valores em falta nas avaliacoes:\n")
print(ratings.isna().sum())

Valores em falta nas avaliacoes:

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### 2- Para maior facilidade visual, juntar-se-ão as avaliações aos títulos


In [29]:
avaliacoes = ratings.merge(
    movies, 
    on="movieId",
    how="left",
    validate="many_to_one"
)

display(avaliacoes[["userId","title","rating"]].head(10))

,userId,title,rating
0,1,Toy Story (1995),4.0
1,1,Grumpier Old Men (1995),4.0
2,1,Heat (1995),4.0
3,1,Seven (a.k.a. Se7en) (1995),5.0
4,1,"Usual Suspects, The (1995)",5.0
5,1,From Dusk Till Dawn (1996),3.0
6,1,Bottle Rocket (1996),5.0
7,1,Braveheart (1995),4.0
8,1,Rob Roy (1995),5.0
9,1,Canadian Bacon (1995),5.0


Apresentam-se as primeiras dez avaliações, associadas aos respetivos títulos dos filmes. Neste conjunto de dados, estas linhas correspondem ao utilizador 1.

**Observação**: o .head(x) anda a buscar os "x" primeiros dados do DataFrame Pandas.

~*Analizando cenários*~ 
Para caso queira se explorar aos filmes correspondentes do userId 1:

In [30]:
avaliacoes_user1 = avaliacoes.loc[
    avaliacoes["userId"] == 1,
    ["title", "rating"]
]

print("Número de filmes avaliados pelo utilizador 1:", len(avaliacoes_user1))
display(avaliacoes_user1.head(10))

Número de filmes avaliados pelo utilizador 1: 232


,title,rating
0,Toy Story (1995),4.0
1,Grumpier Old Men (1995),4.0
2,Heat (1995),4.0
3,Seven (a.k.a. Se7en) (1995),5.0
4,"Usual Suspects, The (1995)",5.0
5,From Dusk Till Dawn (1996),3.0
6,Bottle Rocket (1996),5.0
7,Braveheart (1995),4.0
8,Rob Roy (1995),5.0
9,Canadian Bacon (1995),5.0


### 3- Nosso objectivo é: dado um filme e um utilizador, uma previsão da nota. Serão utilizados dois modelos de Regressão: **Ridge** e **Árvore de Decisão**.

In [31]:
from sklearn.model_selection import train_test_split

# Agora, ao invés de ir tratar o id como números, converteremos para string, para que possa ser categorizado.

X = ratings[["userId","movieId"]].astype(str)

# Resposta que queremos prever

y= ratings["rating"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Avaliações de treino:",len(X_treino))
print("Avaliações de teste:",len(X_teste))

Avaliações de treino: 80668
Avaliações de teste: 20168


### 4 - Preparação das Categorias e Criação dos Modelos

As variáveis de entrada são os identificadores do utilizador (*userId*) e do filme (*movieId*).

Apesar de se tratar de números, representam as categorias. Os números por si só, não trazem nenhum valor. 

Por isso, os identificadores serão transformados através do ***One-Hot-Encoding** antes de serem utilizados por modelos seguintes.

***One-Hot-Encoding**

As categorias agora, serão transformadas em colunas de "sim" (1) ou "não" (0).

Exemplo:

Os identificadores devem de ser representados como categorias e não quantidades:

*Antes*:

|userId | movieId|
|-------|--------|
|1      |     10|
|2      |    20 |
|1      |     20|

*Depois do **One-Hot-Encoding***:



| É o utilizador 1? | É o utilizador 2? | É o filme 10? | É o filme 20? |
|---|---|---|---|
| 1 | 0 | 1 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 |

*Exemplos ilustrativos, apenas*

Evita confusão de interpretar identificadores como quantidades.

### 5 - Uso do OneHotEncoder 

In [32]:
from sklearn.preprocessing import OneHotEncoder

exemplo = X_treino[["userId","movieId"]].head(5).copy()
codificador_exemplo = OneHotEncoder(handle_unknown="ignore")

exemplo_transformado = codificador_exemplo.fit_transform(exemplo)

#trazer em tabela com *pandas*

tabela_exemplo = pd.DataFrame(
    exemplo_transformado.toarray(), columns=codificador_exemplo.get_feature_names_out(), index=exemplo.index
)

print("Identificadores antes:")
display(exemplo)

print("Com One-Hot-Encoding:")
display(tabela_exemplo)

Identificadores antes:


,userId,movieId
80568,509,7347
50582,326,71462
8344,57,2115
99603,610,1127
71701,462,2409


Com One-Hot-Encoding:


,userId_326,userId_462,userId_509,userId_57,userId_610,movieId_1127,movieId_2115,movieId_2409,movieId_71462,movieId_7347
80568,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
50582,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8344,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
99603,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
71701,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


### 5.1 - Pipeline de Regressão **Ridge**

A **Regressão de Ridge** é uma regressão linear com regularização. Além de procurar reduzir o erro das previsões, controla o sobreajuste, ao penalizar coeficientes muito elevados.

De certa forma, o modelo pode representar a tendência de um utilizador atribuir notas mais altas e a de um filme receber melhores avaliações. O parâmetro **alfa** controla a intensidade da regularização e será optmizado posteriormente. É lido o "efeito do utilizador" / "efeito do filme".

#### Nota prevista = valor de base (rating) + efeito do utilizador + efeito do filme

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

modelo_ridge = Pipeline([
    (
        "categorias",
        OneHotEncoder(handle_unknown="ignore")
    ),
    (
        "regressao",
        Ridge(
            alpha=1.0,  # valor inicial de regularização
            solver="lsqr" #metodo de calculo compativel
        )
    )
])

display(modelo_ridge)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorias', ...), ('regressao', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodo

### 5.3 - Pipeline Árvore de Decisão

A Árvore de Decisão será muito útil para guiar as decisões e dividir os dados, a partir de regras sucessivas. A ideia é formar grupos com avaliações semelhantes.

Neste Projecto, as regras serão construídas a partir dos identificadores codificados dos utilizadores e dos filmes. A profundidade máxima e o número mínimo de exemplos por folha permitem o controlo da complexidade da árvore e serão ajustados na fase de optmização.

In [34]:
from sklearn.tree import DecisionTreeRegressor

modelo_arvore = Pipeline([
    (
        "categorias",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    ),
    (
        "regressao",
        DecisionTreeRegressor(
            max_depth=10,   #profundidade da árvore
            min_samples_leaf=20,    #exige 20 avaliações de treino em cada folha
            random_state=42
        )
    )
])
display(modelo_ridge)
display(modelo_arvore)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorias', ...), ('regressao', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodo

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorias', ...), ('regressao', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodo

### 5.4 - Uso das Pipelines

Cada modelo foi incluído numa pipeline que reúna a codificação das categorias e a regressão.

As pipelines aprenderão categorias apenas com a parte usada para treino em cada divisão. A parte da **validação** usará dessa codificação, sem participar da *aprendizagem*. Essa organização mantém o mesmo procedimento de preparação durante o treino e a previsão e ajuda a evitar fuga de dados entre o *treino* e a *validação*.

### 6 - Validação e Optmização dos Modelos

Para comparar os modelos e escolher seus hiperparâmetros, será utilizada a validação cruzada com três divisões no conjunto de treino.

Em cada avaliação, duas partes serão utilizadas para *treinar* e a outra para *validar*. O processo irá correr, até que todas partes tenham passado uma vez pela validação.

In [35]:
from sklearn.model_selection import KFold, GridSearchCV
# KFold (scikit-learn) - tecnica cross-validation machine learning usada para avaliar performance de modelo e generalição ao dar *split* no dataset em K subsets de tamanhos iguais. 
validacao = KFold(
    n_splits=3,     #Treino a ser dividido em tres partes
    shuffle=True,   #Baralha as linhas antes de formar as partes
    random_state=42 #reproduz as mesmas divisões
)

### 6.1 - Optmização Modelo Ridge

Será optimizado o parâmetro *alfa*, que regulariza os coeficientes das variáveis, o que, minimiza o erro entre a previsão e o valor real. **Evita-se**: erro em dados novos e *underfitting*.

#### GridSearchCV

O **GridSearchCV** é uma ferramenta de ML que na utilização abaixo, irá atrás do ajuste do nosso hiperparâmetro *alfa*.

***CV**: Cross Validation 

In [37]:
import pandas as pd
# O parametro utilizado sera o ALFA (Modelo Ridge Scikit-Learn)
parametros_ridge = {
    "regressao__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

busca_ridge = GridSearchCV(     # O GridSearchCV esta a ser util para a organizacao interna das divisoes de validacao.
    estimator=modelo_ridge,
    param_grid=parametros_ridge,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=-1
)
# A partir dos parametros passados, faz-se o treino com os dados
busca_ridge.fit(X_treino,y_treino)

# Imprimimos os melhores parametros encontrados
print("Melhores Parametros:", busca_ridge.best_params_)

# RMSE mede o desempenho dentro do processo de validacao cruzada (CV)
rmse_validacao_ridge=(-busca_ridge.best_score_)**0.5
print(f"RMSE de validacao:{rmse_validacao_ridge:.3f}")

Melhores Parametros: {'regressao__alpha': 1.0}
RMSE de validacao:0.875


### 6.2 - Optmização da Árvore de Decisão

Na Árvore de Decisão, serão optmizadas a profundidade máxima (*max_depth*) e o númemro de avaliações mínimias por folha (*min_samples_leaf*). 

* *max_depth*: limita a profundidade da árvore, quantos níveis de decisão pode ter.
* *min_samples_leaf*: define quantas avaliações de treino, devem ter em cada grupo (folha).

Ainda não sabemos quais os melhores limites para as melhores previsões, por isso tentaremos com profundidade máxima 10 e 20, como no código a seguir, tal qual mínimos de 20 e 100 avaliações por folha.

In [39]:
parametros_arvore = {
    "regressao__max_depth":[10,20],
    "regressao__min_samples_leaf":[20,100]
}

busca_arvore = GridSearchCV(
    estimator=modelo_arvore,
    param_grid=parametros_arvore,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=1
)

busca_arvore.fit(
    X_treino,
    y_treino
)

print("Melhores Parametros:",busca_arvore.best_params_)

rmse_validacao_arvore=(-busca_arvore.best_score_)**0.5

print(f"RMSE de validacao:{rmse_validacao_arvore:.3f}")

Melhores Parametros: {'regressao__max_depth': 20, 'regressao__min_samples_leaf': 20}
RMSE de validacao:0.990


### 6.3 - Comparação dos Resultados

In [40]:
comparacao_validacao = pd.DataFrame([
    {
        "Modelo":"Ridge",
        "RMSE de Validacao":rmse_validacao_ridge,
        "Melhores Parametros":busca_ridge.best_params_
    },
    {
        "Modelo":"Arvore de Decisao",
        "RMSE de Validacao":rmse_validacao_arvore,
        "Melhores Parametros":busca_arvore.best_params_
    }
]).sort_values("RMSE de Validacao").reset_index(drop=True)

display(comparacao_validacao)

,Modelo,RMSE de Validacao,Melhores Parametros
0,Ridge,0.874831,{'regressao__alpha': 1.0}
1,Arvore de Decisao,0.990336,"{'regressao__max_depth': 20, 'regressao__min_s..."
